# Demo: 3D image binary classification  

> SynapseMNIST3D dataset demo


In [ ]:
#| default_exp demo

In [1]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.core import *
from bioMONAI.core import Path
from bioMONAI.data import get_target, RandomSplitter
#from bioMONAI.nets import BasicUNet, DynUNet
from bioMONAI.losses import *
from bioMONAI.metrics import *
from bioMONAI.datasets import *
from bioMONAI.visualize import *

import medmnist
import os

#from monai.utils import set_determinism
from monai.transforms import ScaleIntensity

#set_determinism(0)

In [2]:
device = get_device()
print(device)

cuda


### Download and store the dataset

In [3]:
data_flag = 'synapsemnist3d'
data_path = Path('../_data/medmnist_data/')

# Ensure the save directory exists; create it if not
os.makedirs(data_path, exist_ok=True)

train_data, val_data, test_data = download_medmnist(data_flag,[],data_path)

# print training and validation datasets
print(train_data)
print("===================")
print(val_data)

Using downloaded and verified file: ../_data/medmnist_data/synapsemnist3d.npz
Using downloaded and verified file: ../_data/medmnist_data/synapsemnist3d.npz
Using downloaded and verified file: ../_data/medmnist_data/synapsemnist3d.npz
Dataset SynapseMNIST3D of size 28 (synapsemnist3d)
    Number of datapoints: 1230
    Root location: ../_data/medmnist_data
    Split: train
    Task: binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'inhibitory synapse', '1': 'excitatory synapse'}
    Number of samples: {'train': 1230, 'val': 177, 'test': 352}
    Description: The SynapseMNIST3D is a new 3D volume dataset to classify whether a synapse is excitatory or inhibitory. It uses a 3D image volume of an adult rat acquired by a multi-beam scanning electron microscope. The original data is of the size 100×100×100um^3 and the resolution 8×8×30nm^3, where a (30um)^3 sub-volume was used in the MitoEM dataset with dense 3D mitochondria instance segmentation labels. Three neuroscience 

In [4]:
train_path = data_path/'train'
val_path = data_path/'val'

train_data.save(train_path)
val_data.save(val_path)

Saving train set to ../_data/medmnist_data/train/synapsemnist3d, csv_path=../_data/medmnist_data/train/synapsemnist3d.csv...


100%|██████████| 1230/1230 [00:32<00:00, 37.49it/s]


Saving val set to ../_data/medmnist_data/val/synapsemnist3d, csv_path=../_data/medmnist_data/val/synapsemnist3d.csv...


100%|██████████| 177/177 [00:04<00:00, 39.19it/s]


In [16]:
from fastai.vision.all import CategoryBlock, get_image_files, GrandparentSplitter, parent_label


batch_size = 128

data_ops = {
    'blocks':       (BioImageBlock(cls=BioImageBase), CategoryBlock),
    'get_items':    get_image_files,
    'splitter':     GrandparentSplitter(train_name=train_path/'synapsemnist3d', valid_name=val_path/'synapsemnist3d'),
    'get_y':        parent_label,
    'item_tfms':    [ScaleIntensity()],
    'bs':           batch_size,
}

In [14]:
data = get_dataloader(
    train_path/'synapsemnist3d',
    show_summary=True,
    **data_ops,
)

Setting affine, but the applied meta contains an affine. This will be overwritten.


Setting-up type transforms pipelines
Found 1230 items
2 datasets of sizes 984,246
Setting up Pipeline: BioImageBase.create
Setting up Pipeline: parent_label -> Categorize -- {'vocab': None, 'sort': True, 'add_na': False}

Building one sample
  Pipeline: BioImageBase.create
    starting from
      ../_data/medmnist_data/train/synapsemnist3d/train628_1.gif
    applying BioImageBase.create gives
      BioImageBase of size 28x28x28x3
  Pipeline: parent_label -> Categorize -- {'vocab': None, 'sort': True, 'add_na': False}
    starting from
      ../_data/medmnist_data/train/synapsemnist3d/train628_1.gif
    applying parent_label gives
      synapsemnist3d
    applying Categorize -- {'vocab': None, 'sort': True, 'add_na': False} gives
      TensorCategory(0)

Final sample: (BioImageBase([[[[141., 141., 141.],
          [  7.,   7.,   7.],
          [ 37.,  37.,  37.],
          ...,
          [ 95.,  95.,  95.],
          [197., 197., 197.],
          [157., 157., 157.]],

         [[ 13.,  

Setting affine, but the applied meta contains an affine. This will be overwritten.



Applying batch_tfms to the batch built
  Pipeline: Tensor2BioImage -- {}
    starting from
      (MetaTensor of size 4x28x28x28x3, metatensor([0., 0., 0., 0.], device='cuda:0'))
    applying Tensor2BioImage -- {} gives
      (BioImageBase of size 4x28x28x28x3, metatensor([0., 0., 0., 0.], device='cuda:0'))
None


Setting affine, but the applied meta contains an affine. This will be overwritten.


In [ ]:
# visualization
mosaic_image_3d(data[0],cmap='gray')

In [ ]:
data.show_batch(max_n=2, cmap='hot')

### Load and train a 3D model

In [ ]:
import torch
from monai.networks.nets import DenseNet121
from torchmetrics.classification import BinaryAccuracy

net = DenseNet121(spatial_dims=3, in_channels=1, out_channels=1)

loss = torch.nn.CrossEntropyLoss()
metric = BinaryAccuracy()

trainer = fastTrainer(data, net, loss_fn=loss, metrics=metric, show_summary=True)

In [ ]:
trainer.fit_flat_cos(500)

In [ ]:
trainer.show_results(cmap='gray')

In [ ]:
# trainer.save('tmp-model')

### Test data 
Evaluate the performance of the selected model on unseen data.
It’s important to not touch this data until you have fine tuned your model to get an unbiased evaluation!